In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Vizualizace časování obvodů

<Accordion>
<AccordionItem title="Verze balíčků">

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Zatímco [nástroj pro kreslení časové osy](/guides/visualize-circuit-timing) zabudovaný v Qiskit je užitečný pro statické obvody, nemusí přesně odrážet časování [dynamických obvodů](/guides/classical-feedforward-and-control-flow) kvůli implicitním operacím, jako je vysílání a určení větví. Jako součást podpory dynamických obvodů vrací Qiskit Runtime přesné informace o časování obvodu uvnitř výsledků úlohy, pokud je to vyžádáno.

> **Note:** - Toto je experimentální funkce. Je ve stavu preview release a podléhá tedy změnám.
> - Tato funkce se vztahuje pouze na úlohy Qiskit Runtime Sampler.
> - Ačkoli je celkový čas obvodu vrácen v metadatech „compilation", NENÍ to čas používaný pro fakturaci (čas QPU).
### Povolení načítání dat o časování
Chceš-li povolit načítání dat o časování, nastav experimentální příznak `scheduler_timing` na `True` při spouštění primitivní úlohy.

In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Přístup k datům časování obvodu
Pokud je to vyžádáno, jsou data časování obvodu pro každý PUB vrácena v metadatech výsledku úlohy pod `["compilation"]["scheduler_timing"]["timing"]`. Toto pole obsahuje nezpracované informace o časování. Chceš-li zobrazit informace o časování, použij vestavěný nástroj pro vizualizaci, jak je popsáno v části [Vizualizace časování](#visualize-timings).

Pomocí následujícího kódu přistupuješ k datům časování obvodu pro první PUB:

In [2]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Pochopení nezpracovaných dat časování
Zatímco vizualizace dat časování obvodu pomocí metody `draw_circuit_schedule_timing` je nejběžnějším případem použití, může být užitečné pochopit strukturu vrácených nezpracovaných dat časování. To by ti mohlo pomoci například extrahovat informace programově.

Data časování vrácená v `["compilation"]["scheduler_timing"]["timing"]` jsou seznam řetězců. Každý řetězec představuje jednu instrukci na nějakém kanálu a je oddělen čárkami na následující datové typy:

- `Branch` - Určuje, zda je instrukce v řídicím toku (then / else) nebo hlavní větvi.
- `Instruction` - Gate a Qubit, na kterém se operuje.
- `Channel` - Kanál, který je přiřazen s instrukcí. Může být jeden z následujících:
      - `Qubit x` - Řídicí kanál pro Qubit _x_.
      - `AWGRx_y` (čtení z libovolného generátoru průběhů) - Používán čtecími kanály ke komunikaci při měření Qubitů. Argumenty _x_ a _y_ odpovídají ID čtecího přístroje a číslu Qubitu.
- `T0` - Čas začátku instrukce v rámci celého harmonogramu
- `Duration` - Doba trvání instrukce v jednotkách _dt_ sekund, kde 1 dt = 1 plánovací cyklus. Hodnotu `dt` backendu najdeš pomocí [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` - Typ pulzní operace, která se používá.

Příklad:

In [3]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### Vizualizace časování
S `qiskit-ibm-runtime` v0.43.0 nebo novějším můžeš vizualizovat časování obvodů. Chceš-li vizualizovat časování, musíš nejprve převést metadata výsledku na `fig` pomocí [metody `draw_circuit_schedule_timing`](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26). Tato metoda vrací figuru `plotly`, kterou můžeš zobrazit přímo, uložit do souboru nebo obojí. Více informací o příkazech `plotly`, které se mají použít, najdeš v [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) a [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html).

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d8287kugbeec73alfbug (DONE)


![Najetí myší na výstup zobrazuje informace jako čas začátku, konce a trvání.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Příklad vygenerované figury')
#### Pochopení vygenerované figury
Obrázek výstupu dat časování obvodu z `draw_circuit_schedule_timing` přenáší následující informace:

- Osa X je čas v jednotkách _dt_ sekund, kde 1 dt = 1 plánovací cyklus. Hodnotu `dt` backendu najdeš pomocí [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- Osa Y je kanál (představ si kanály jako nástroje, které vysílají pulzy).
    - `Receive channel` - Jediný kanál, který není sám o sobě nástrojem. Je to instrukce přehrávaná na všech kanálech, které jsou v danou dobu součástí komunikačního postupu s hubem.
    - `Qubit x` - Řídicí kanál pro Qubit x.
    - `AWGRx_y` (čtení z libovolného generátoru průběhů) - Používán čtecími kanály ke komunikaci při měření Qubitů. Argumenty _x_ a _y_ odpovídají ID čtecího přístroje a číslu Qubitu.
    - `Hub` - Řídí vysílání.

Každá instrukce má navíc formát *X_Y*, kde *X* je název instrukce a *Y* je typ pulzu. Typ `play` aplikuje řídicí pulzy a `capture` zaznamenává stav Qubitu. Můžeš také najet myší na každou instrukci a získat více podrobností. Předchozí obrázek například ukazuje řídicí pulz pro Gate X aplikovaný na Qubit 10 v čase 1161 dt.
### Komplexní příklad
Tento příklad ukazuje, jak povolit tuto možnost, získat informace o časování obvodu z metadat a zobrazit je jako obrázek.

Nejprve nastav prostředí, definuj obvody a převeď je na obvody ISA a definuj a spusť úlohy.

In [5]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,1365,0,shift_phase\nmain,sx_0,Qubit 0,1365,9,play\nmain,sx_0,Qubit 0,1369,0,shift_phase\nmain,rz_0,Qubit 0,1374,0,shift_phase\nmain,barrier,Qubit 0,1374,0,barrier\nmain,measure_0,Qubit 0,1374,64,play\nmain,measure_0,Qubit 0,1438,108,play\nmain,measure_0,AWGR0_0,1485,360,capture\nmain,measure_0,Qubit 0,1546,64,play\nmain,measure_0,Qubit 0,1610,64,play\nmain,barrier,Qubit 0,2046,0,barrier\nmain,broadcast,Hub,1485,561,broadcast\nmain,receive,Receive,2046,7,receive\nthen,x_0,Qubit 0,2061,9,play\nmain,barrier,Qubit 0,2079,0,barrier\nmain,measure_0,Qubit 0,2079,64,play\nmain,measure_0,Qubit 0,2143,108,play\nmain,measure_0,AWGR0_0,2190,360,capture\nmain,measure_0,Qubit 0,2251,64,play\nmain,measure_0,Qubit 0,2315,64,play\nmain,barrier,Qubit 0,2725,0,barrier\nmain,barrier,Qubit 0,2725,0,barrier\n'

Finally, you can visualize and save the timing:

In [6]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Dále získej časování harmonogramu obvodu: